In [1]:
#Let's load everything
import pandas as pd
import numpy as np

rfm = pd.read_csv("../data/processed/rfm_features.csv")
campaign_design = pd.read_csv("../data/processed/campaign_design.csv")

segments = rfm.groupby("segment_name").agg(
    n_customers=("customer_unique_id", "count"),
    avg_order_value=("avg_order_value", "mean"),
).reset_index()

segments = segments.merge(campaign_design, on="segment_name")
segments

,segment_name,n_customers,avg_order_value,channel,offer_type,strategy
0,Casual Low-Spend Majority,46437,54.794206,Email,Free shipping threshold / small % off,Low-cost broad reach
1,Dissatisfied & Disengaged,13747,122.188100,Direct email,Service recovery credit,"Retention/recovery, not promotional"
2,High-Value Financed One-Timers,30950,267.642353,Email + retargeting,Free shipping / extended installments,Cross-sell into new categories
3,Loyal High-Value Repeaters,2224,148.276579,Personalized email,Loyalty tier / early access,"Recognition, deepen relationship"


In [2]:
#response rates
baseline_response_rate = 0.0016  # Klaviyo 2026 ecommerce placed-order rate

segment_multiplier_map = {
    "High-Value Financed One-Timers": 1.1,
    "Dissatisfied & Disengaged": 0.4,
    "Casual Low-Spend Majority": 0.8,
    "Loyal High-Value Repeaters": 2.5,
}

segments["segment_multiplier"] = segments["segment_name"].map(segment_multiplier_map)
segments["response_rate"] = baseline_response_rate * segments["segment_multiplier"]

In [3]:
#campaign cost per customer, tied to the channel each segment actually gets
# Document your cost assumptions — these should reflect real per-contact costs for each channel type
channel_cost_map = {
    "Email + retargeting": 0.20,      # email (~$0.01-0.05) + display retargeting media cost
    "Direct email": 0.05,             # plain transactional-style email, no ad spend
    "Email": 0.03,                    # simple bulk email, lowest cost
    "Personalized email": 0.15,       # more design/copy effort per segment, still email-only delivery
}

segments["campaign_cost_per_customer"] = segments["channel"].map(channel_cost_map)
segments[["segment_name", "channel", "campaign_cost_per_customer"]]

,segment_name,channel,campaign_cost_per_customer
0,Casual Low-Spend Majority,Email,0.03
1,Dissatisfied & Disengaged,Direct email,0.05
2,High-Value Financed One-Timers,Email + retargeting,0.20
3,Loyal High-Value Repeaters,Personalized email,0.15


In [ ]:
#the core simulation math
